### HCP Primary HCO

In [0]:
SELECT * FROM com_edp_prd.com_raw.kom_pharmacy_events
where ndc11 like '54092070001';

In [0]:
CREATE OR REPLACE TEMPORARY VIEW specialty_groupings AS
SELECT * FROM (
    VALUES
        ('Neurology Specialists', 'Psychiatry & Neurology'),
        ('Neurology Specialists', 'Neurological Surgery'),
        ('Neurology Specialists', 'Surgery'),
        ('Genetic Experts', 'Medical Genetics'),
        ('Referral & Early Diagnosis Staff', 'Pediatrics'),
        ('Referral & Early Diagnosis Staff', 'Family Medicine'),
        ('Referral & Early Diagnosis Staff', 'Internal Medicine'),
        ('Referral & Early Diagnosis Staff', 'Physician Assistant'),
        ('Care Coordination & Nursing', 'Nurse Practitioner'),
        ('Systemic Symptom Specialists', 'Orthopaedic Surgery'),
        ('Systemic Symptom Specialists', 'Otolaryngology'),
        ('Systemic Symptom Specialists', 'Ophthalmology'),
        ('Perioperative & Procedural Specialists', 'Nurse Anesthetist, Certified Registered'),
        ('Perioperative & Procedural Specialists', 'Anesthesiology'),
        ('Diagnostics & Laboratory', 'Radiology'),
        ('Diagnostics & Laboratory', 'Pathology'),
        ('Acute / Inpatient Care', 'Emergency Medicine'),
        ('Acute / Inpatient Care', 'Hospitalist'),
        ('Rehabilitation & Supportive Care', 'Physical Medicine & Rehabilitation')
) AS t(mapped_bucket, specialty);


In [0]:
-- Elaprase Treated patients (Can or cannot be mpsii diagnosed)
-- A single patient can be attributed to multiple HCPs

create or replace temporary view tx_claims as
with elaprase_tx_claims as (SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 RENDERING_NPI AS NPI,
                 REFERRING_NPI AS REFERRING_NPI,
                 NDC11 AS CODE,
                 MEDICAL_EVENT_ID AS EVENT_ID,
                 SERVICE_DATE AS FILL_DATE,
                 PLACE_OF_SERVICE,
                 KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME

         FROM com_edp_prd.com_raw.kom_medical_events 
         WHERE NDC11 IN ('54092070001','540920700')
    UNION
         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                 PRESCRIBER_NPI AS NPI,
                 NULL AS REFERRING_NPI,
                 NDC11 AS CODE,
                 PHARMACY_EVENT_ID as EVENT_ID,
                 FILL_DATE,
                 NULL AS PLACE_OF_SERVICE,
                 COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
                 'PHARMACY_EVENTS' AS TABLE_NAME
           FROM com_edp_prd.com_raw.kom_pharmacy_events
           WHERE NDC11 IN ('54092070001','540920700')
           AND TRANSACTION_RESULT = 'PAID'
    UNION

         SELECT DISTINCT PATIENT_ID AS PATIENT_ID,
                RENDERING_NPI AS NPI,
                REFERRING_NPI AS REFERRING_NPI,
                PROCEDURE_CODE AS CODE,   
                MEDICAL_EVENT_ID AS EVENT_ID,
                SERVICE_DATE AS FILL_DATE, 
                PLACE_OF_SERVICE,
                KH_PLAN_ID AS KH_PLAN,
                'MEDICAL_EVENTS' AS TABLE_NAME


           FROM com_edp_prd.com_raw.kom_medical_events 
           WHERE PROCEDURE_CODE IN ('99601','99602','96365','96366','J1743','S9357','S9379','38206','38230','38232','38240','38241','38242','38243','38250')),
specialty_info as (
  select a.*, b.PRIMARY_SPECIALTY as specialty, 
  CASE 
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NULL THEN NULL
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NOT NULL THEN 'other'
            ELSE c.mapped_bucket
  END AS specialty_bucket, 
  pos.description as pos_description, d.HCO_PRIMARY_NPI as hco_npi_thm
  from elaprase_tx_claims as a 
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.NPI and b.PROVIDER_TYPE = 'INDIVIDUAL'
  left join specialty_groupings as c on b.PRIMARY_SPECIALTY = c.specialty
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
  ON TRY_CAST(NULLIF(TRIM(A.PLACE_OF_SERVICE), '') AS INT) = pos.code
  left join com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping as d
  on a.npi = d.hcp_npi
),
tx_claims_5_years as (
  select *
from specialty_info
where fill_date between '2020-08-01' and '2025-07-31'
),
hcp_vid AS (
  -- get HCP entity VID from vod_hcp so we can join to parent HCO relationships
  SELECT
    TRY_CAST(npi_num__v AS BIGINT) AS hcp_npi,
    vid__v AS hcp_vid
  FROM com_edp_prd.com_raw.vod_hcp
  WHERE npi_num__v IS NOT NULL
),
affiliations AS (
  SELECT
    b.hcp_npi,
    c.PARENT_HCO_VID__V,
    c.modified_date__v,
    c.status_update_time__v
  FROM tx_claims_2_years a
  JOIN hcp_vid b
    ON a.npi = b.hcp_npi
  JOIN com_edp_prd.com_raw.vod_parenthco c
    ON b.hcp_vid = c.ENTITY_VID__V and c.PARENT_HCO_STATUS__V = 'A' AND c.RELATIONSHIP_TYPE__V = '7356' and c.HIERARCHY_TYPE__V = 'HCP_HCO'
),
ranked AS (
  SELECT
    a.hcp_npi,
    b.npi_num__v AS hco_npi,
    ROW_NUMBER() OVER (
      PARTITION BY a.hcp_npi
      ORDER BY a.modified_date__v DESC NULLS LAST, a.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM affiliations a
  LEFT JOIN com_edp_prd.com_raw.vod_hco b
    ON a.PARENT_HCO_VID__V = b.vid__v
),
top_ranked_hco as (SELECT
  hcp_npi,
  TRY_CAST(hco_npi AS BIGINT) AS hco_npi_vod
FROM ranked
WHERE rn = 1 and hco_npi is not null
ORDER BY hcp_npi)
SELECT
  -- explicit columns from tx_claims_2_years (add/remove columns here as needed)
  a.PATIENT_ID,
  a.NPI,
  a.referring_npi,
  a.CODE,
  a.EVENT_ID,
  a.FILL_DATE,
  a.PLACE_OF_SERVICE,
  a.KH_PLAN,
  a.TABLE_NAME,
  a.specialty,
  a.specialty_bucket,
  a.pos_description,
  COALESCE(TRY_CAST(a.hco_npi_thm AS BIGINT), b.hco_npi_vod) AS primary_hco
FROM tx_claims_2_years a
LEFT JOIN top_ranked_hco b
  ON a.npi = b.hcp_npi;

In [0]:
select * from tx_claims limit 4;

In [0]:
create or replace temporary view dx_claims as
with diagnosis_claims as (
  SELECT DISTINCT 
      PATIENT_ID,
      RENDERING_NPI AS NPI,
      SERVICE_DATE AS FILL_DATE,
      MEDICAL_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODES,
      KH_PLAN_ID AS KH_PLAN,
      Place_of_service
    FROM com_edp_prd.com_raw.kom_medical_events
    WHERE DIAGNOSIS_CODES ilike '%E761%'

    UNION

    -- Pharmacy Events
    SELECT DISTINCT 
      PATIENT_ID,
      PRESCRIBER_NPI AS NPI,
      FILL_DATE,
      PHARMACY_EVENT_ID AS EVENT_ID,
      DIAGNOSIS_CODE AS DIAGNOSIS_CODES,
      COALESCE(PRIMARY_KH_PLAN_ID, SECONDARY_KH_PLAN_ID) AS KH_PLAN,
      NULL as Place_of_service
    FROM com_edp_prd.com_raw.kom_pharmacy_events
    WHERE DIAGNOSIS_CODE ILIKE '%E761%'
      AND TRANSACTION_STATUS = 'PAID'),
specialty_info as (
  select a.*, b.primary_specialty as specialty, 
  CASE 
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NULL THEN NULL
            WHEN c.mapped_bucket IS NULL AND b.PRIMARY_SPECIALTY IS NOT NULL THEN 'other'
            ELSE c.mapped_bucket
  END AS specialty_bucket, 
  pos.description as pos_description, d.HCO_PRIMARY_NPI as hco_npi_thm
  from diagnosis_claims as a
  left join com_edp_prd.com_raw.kom_providers as b on a.npi = b.npi and b.provider_type = 'INDIVIDUAL'
  left join specialty_groupings as c on b.PRIMARY_SPECIALTY = c.specialty
  LEFT JOIN com_edp_prd.cmpa_insights_internal_schema.pos_description pos
  ON TRY_CAST(NULLIF(TRIM(A.PLACE_OF_SERVICE), '') AS INT) = pos.code
  left join com_edp_prd.cmpa_insights_internal_schema.target_hcp_hco_mapping as d
  on a.npi = d.hcp_npi
),
dx_claims_5_years as (select * from specialty_info
where fill_date between '2020-08-01' AND '2025-07-31'),
hcp_vid AS (
  -- get HCP entity VID from vod_hcp so we can join to parent HCO relationships
  SELECT
    TRY_CAST(npi_num__v AS BIGINT) AS hcp_npi,
    vid__v AS hcp_vid
  FROM com_edp_prd.com_raw.vod_hcp
  WHERE npi_num__v IS NOT NULL
),
affiliations AS (
  SELECT
    b.hcp_npi,
    c.PARENT_HCO_VID__V,
    c.modified_date__v,
    c.status_update_time__v
  FROM dx_claims_5_years a
  JOIN hcp_vid b
    ON a.npi = b.hcp_npi
  JOIN com_edp_prd.com_raw.vod_parenthco c
    ON b.hcp_vid = c.ENTITY_VID__V and c.PARENT_HCO_STATUS__V = 'A' AND c.RELATIONSHIP_TYPE__V = '7356' and c.HIERARCHY_TYPE__V = 'HCP_HCO'
),
ranked AS (
  SELECT
    a.hcp_npi,
    b.npi_num__v AS hco_npi,
    ROW_NUMBER() OVER (
      PARTITION BY a.hcp_npi
      ORDER BY a.modified_date__v DESC NULLS LAST, a.status_update_time__v DESC NULLS LAST
    ) AS rn
  FROM affiliations a
  LEFT JOIN com_edp_prd.com_raw.vod_hco b
    ON a.PARENT_HCO_VID__V = b.vid__v
),
top_ranked_hco as (SELECT
  hcp_npi,
  TRY_CAST(hco_npi AS BIGINT) AS hco_npi_vod
FROM ranked
WHERE rn = 1 and hco_npi is not null
ORDER BY hcp_npi)
SELECT
  -- explicit columns from tx_claims_2_years (add/remove columns here as needed)
  a.PATIENT_ID,
  a.NPI,
  a.FILL_DATE,
  a.EVENT_ID,
  a.DIAGNOSIS_CODES,
  a.KH_PLAN,
  a.PLACE_OF_SERVICE,
  a.specialty,
  a.specialty_bucket,
  a.pos_description,
  COALESCE(TRY_CAST(a.hco_npi_thm AS BIGINT), b.hco_npi_vod) AS primary_hco
FROM dx_claims_5_years a
LEFT JOIN top_ranked_hco b
  ON a.npi = b.hcp_npi;

In [0]:
select * from dx_claims limit 4;

**Individual Cohort Numbers :-**

| **Factors** | **Unique Patient Count** | **Have a Tx/Dx Claim** |
| --- | --- | --- |
| **Dx Claims (5 Years)** | 2928 | 535 |
| **Tx Claims (2 Years)** | 3759 | 535 (1 Dx Claim), 401 (2 Dx Claim) |
| **Overall Claims** | 6152 | &nbsp; |

In [0]:
with overall_cohort as (select distinct 'dx_claim' as claim_type, patient_id, npi, specialty, specialty_bucket
from dx_claims
union
select distinct 'tx_claim' as claim_type, patient_id, npi, specialty, specialty_bucket
from tx_claims)
select specialty_bucket, count(distinct patient_id) as patient_count
from overall_cohort
where npi is not null and specialty is not null
group by 1 order by 2 desc

In [0]:
select place_of_service, pos_description, count(distinct patient_id) as patient_count
from tx_claims
where place_of_service is not null
group by 1,2 order by 3 desc

In [0]:
with hco_type_info as (select a.*, b.hco_type__v as hco_type, c.name as hco_type_info
from tx_claims as a
left join com_edp_prd.com_raw.vod_hco as b
on a.primary_hco = try_cast(b.npi_num__v as BIGINT)
left join com_edp_prd.com_raw.vod_references as c
on b.hco_type__v = c.code and c.reference_type = 'HCOType')
select hco_type_info, count(distinct patient_id) as patient_count 
from hco_type_info
-- where hco_type_info is not null
group by 1 order by 2 desc


In [0]:
SELECT
    NPI AS rendering_npi, specialty,
    COUNT(DISTINCT PATIENT_ID) AS overall_patient_count,
    COUNT(DISTINCT referring_npi) AS referring_hcp_count,
    COUNT(DISTINCT CASE WHEN referring_npi IS NOT NULL THEN PATIENT_ID END) AS referring_hcp_patient_count
FROM tx_claims
where npi is not null
GROUP BY 1, 2
order by 4 desc, 5 desc, 3 desc;

### Creating HCP Level Table (Initial Target List)

In [0]:
select * from dx_claims limit 4;

In [0]:
select count(distinct npi) from tx_claims where npi in (select distinct npi from dx_claims)

In [0]:
with elaprase_patients as (
  select a.npi, a.specialty, a.specialty_bucket, a.primary_hco, c.name as hco_type_info, count(distinct patient_id) as elaprase_patient_counts
  from tx_claims as a
  left join com_edp_prd.com_raw.vod_hco as b
  on a.primary_hco = try_cast(b.npi_num__v as BIGINT)
  left join com_edp_prd.com_raw.vod_references as c
  on b.hco_type__v = c.code and c.reference_type = 'HCOType'
  where a.npi is not null
  group by 1,2,3,4,5 order by 6 desc
),
first_diagnosis as (
  select patient_id, min(fill_date) as first_diagnosis
  from dx_claims
  group by 1
),
dx_claims_with_first_diagnosis as (
  select a.*, case when b.patient_id is not null then 1 else 0 end as first_diagnosis_flag
  from dx_claims as a
  left join first_diagnosis as b
  on a.patient_id = b.patient_id and a.fill_date = b.first_diagnosis
),
mpsii_diagnosed_patients as (
  select npi, count(distinct patient_id) as mpsii_diagnosed_patients, 
  count(distinct case when first_diagnosis_flag = 1 then patient_id end) as mpsii_diagnosed_patients_being_first_treater
  from dx_claims_with_first_diagnosis
  where npi is not null
  group by 1 order by 2 desc, 3 desc
),
mpsii_diagnosed_patients_with_treatment as (
  select npi, count(distinct patient_id) as mpsii_diagnosed_and_treated_patients
  from tx_claims where patient_id in (select distinct patient_id from dx_claims)
  group by 1 order by 2 desc
),
referral_counts as (
  SELECT
    NPI AS rendering_npi,
    COUNT(DISTINCT referring_npi) AS referring_hcp_count,
    COUNT(DISTINCT CASE WHEN referring_npi IS NOT NULL THEN PATIENT_ID END) AS referring_hcp_patient_count
FROM tx_claims
where npi is not null
GROUP BY NPI
)
select a.*, b.mpsii_diagnosed_patients, b.mpsii_diagnosed_patients_being_first_treater, c.mpsii_diagnosed_and_treated_patients, d.referring_hcp_count, d.referring_hcp_patient_count
from elaprase_patients as a
left join mpsii_diagnosed_patients as b on a.npi = b.npi
left join mpsii_diagnosed_patients_with_treatment as c on a.npi = c.npi
left join referral_counts as d on a.npi = d.rendering_npi
order by a.elaprase_patient_counts desc

In [0]:
select *
from com_edp_prd.com_raw.kom_medical_events
where NDC11 in ('540920700')